# Roll-disturbance testing (interactive)

Same idea as `sim_remote_control.ipynb`'s arrow-key drive control, but for testing roll
recovery instead of driving: **Left/Right arrows inject a real +3deg/-3deg roll kick** into
the running simulation (rotating the chassis's current orientation, not just at reset - the
same technique used to make `01-test-model-in-mujoco/solution_simple_pid.ipynb` start
pre-tilted), while the actual trained stage-05 policy (phase-10's checkpoint) tries to
recover in real time.

- **Left arrow**: kick roll by -3deg
- **Right arrow**: kick roll by +3deg
- **Backspace**: reset the sim (MuJoCo's own binding)

Why 3 degrees specifically: `05-command-conditioning/README.md`'s "Phase 11 outcome"
section calculated a hard physical ceiling at **~3.92deg of roll** - past that angle,
gravity's tipping torque exceeds the reaction wheel's maximum possible torque (using the
motor's only verified spec, its 0.03N.m rated torque), regardless of policy quality. A
kick to 3deg sits right at the edge of that ceiling - close enough to be a real, meaningful
test of the recovery envelope, without being a kick to an angle that's already
mathematically unrecoverable no matter what.

`cmd_vel` is fixed at 0 the whole time (station-keeping only) - isolates the roll-recovery
behavior from any driving motion, same reasoning as the isolated visual demo in
`05-command-conditioning/diagnostics/visual_roll_disturbance_demo.py`.

In [1]:
# Import standard libraries
import math
from pathlib import Path
import sys
import time

# Third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer
import numpy as np
import torch

sys.path.append("..")
from rl.ppo_trainer import PPOConfig, Agent

In [2]:
# Settings
MJCF_PATH = Path("../../mechanical/urdf/uniwheel/uniwheel.xml")

# Stage 05's final checkpoint (phase-10 - see 05-command-conditioning/README.md's
# "Phase 11 outcome" section for why this, not a later phase, is the final policy)
CKPT_PATH = Path(
    "../05-command-conditioning/runs/"
    "UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/best_model.cleanrl_model"
)

KICK_DEG = 1.0         # magnitude of each Left/Right roll kick, in degrees
ALPHA = 0.98           # complementary filter weight - MUST match uniwheel_env_cmd.py
TIP_THRESHOLD_DEG = 30.0
PRINT_EVERY = 25

# Actuator/sensor names (from the MJCF file)
WHEEL_MOTOR = "wheel_motor"
REACTION_WHEEL_MOTOR = "reaction_wheel_motor"
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
WHEEL_VEL = "wheel_vel"
REACTION_WHEEL_VEL = "reaction_wheel_vel"

# Keys
KEY_BACKSPACE = 259
KEY_RIGHT = 262
KEY_LEFT = 263

In [3]:
# Global: a pending roll kick (in degrees) set by key_callback, applied and cleared by the
# main loop on its next iteration (key_callback shouldn't touch MuJoCo state directly -
# same pattern sim_remote_control.ipynb uses for cmd_vel)
pending_kick = {"deg": 0.0}

def key_callback(keycode):
    """Handle a key press"""
    if keycode == KEY_LEFT:
        pending_kick["deg"] = -KICK_DEG
    elif keycode == KEY_RIGHT:
        pending_kick["deg"] = KICK_DEG
    # Backspace isn't handled here - MuJoCo's viewer resets the sim on its own binding,
    # detected in the main loop the same way sim_remote_control.ipynb does

In [4]:
# Load the MuJoCo model
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))
data = mujoco.MjData(model)

wheel_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, WHEEL_MOTOR)
reaction_wheel_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, REACTION_WHEEL_MOTOR)
print(f"Wheel motor ID: {wheel_motor_id}")
print(f"Reaction wheel motor ID: {reaction_wheel_motor_id}")

Wheel motor ID: 0
Reaction wheel motor ID: 1


In [5]:
# Load the trained policy (stage 05, phase-10 - 7 observation inputs, includes cmd_vel)
class _Spaces:
    pass
dummy_envs = _Spaces()
dummy_envs.single_observation_space = type("S", (), {"shape": (7,)})()
dummy_envs.single_action_space = type("S", (), {"shape": (2,)})()

ppo_config = PPOConfig(actor_hidden_layers=2, actor_hidden_size=48,
                        critic_hidden_layers=2, critic_hidden_size=48)
agent = Agent(dummy_envs, ppo_config)
agent.load_state_dict(torch.load(CKPT_PATH, map_location="cpu", weights_only=True))
agent.eval()
print(f"Loaded policy from {CKPT_PATH}")

Loaded policy from ../05-command-conditioning/runs/UniWheel-v0__uniwheel-ppo-cmd-phase-10__42__1788754064/best_model.cleanrl_model


In [6]:
# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Complementary-filter state
pitch = 0.0
roll = 0.0
prev_time = 0.0
steps = 0

with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
    # Define free-look camera (control with mouse) - front-on view makes a roll kick
    # (sideways tip) most visible
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.08]
    viewer.cam.distance = 0.8
    viewer.cam.azimuth = 0
    viewer.cam.elevation = -10

    while viewer.is_running():
        step_start = time.time()

        # Detect MuJoCo reset (backspace) and reset filter state too
        if data.time < prev_time:
            pitch = 0.0
            roll = 0.0
            mujoco.mj_forward(model, data)
        prev_time = data.time

        # Apply a pending roll kick, if Left/Right was just pressed - rotates the chassis's
        # CURRENT orientation (not just at reset) by kick_deg about world X (the roll axis),
        # composed via quaternion multiplication so it stacks correctly with whatever
        # orientation the robot is already in (mid-recovery, mid-fall, etc.)
        if pending_kick["deg"] != 0.0:
            kick_rad = math.radians(pending_kick["deg"])
            kick_quat = np.array([math.cos(kick_rad / 2), math.sin(kick_rad / 2), 0.0, 0.0])
            current_quat = data.qpos[3:7].copy()
            new_quat = np.zeros(4)
            mujoco.mju_mulQuat(new_quat, kick_quat, current_quat)
            data.qpos[3:7] = new_quat
            mujoco.mj_forward(model, data)
            pending_kick["deg"] = 0.0

        # Read sensors - same complementary filter as uniwheel_env_cmd.py's _get_obs()
        accel_x, accel_y, accel_z = data.sensor(IMU_ACCEL).data
        pitch_rate = data.sensor(IMU_GYRO).data[1]
        roll_rate = data.sensor(IMU_GYRO).data[0]

        accel_pitch = -1 * math.atan2(accel_x, accel_z)
        accel_roll = math.atan2(accel_y, accel_z)
        pitch = ALPHA * (pitch + pitch_rate * model.opt.timestep) + (1 - ALPHA) * accel_pitch
        roll = ALPHA * (roll + roll_rate * model.opt.timestep) + (1 - ALPHA) * accel_roll

        wheel_vel = data.sensor(WHEEL_VEL).data[0]
        reaction_wheel_vel = data.sensor(REACTION_WHEEL_VEL).data[0]

        # Build observation and run the policy (deterministic action = actor_mean, for a
        # stable, repeatable test rather than the stochastic training policy). cmd_vel is
        # fixed at 0 - station-keeping only, isolates roll recovery from driving motion.
        obs = np.array([pitch, roll, pitch_rate, roll_rate, wheel_vel,
                         reaction_wheel_vel, 0.0], dtype=np.float32)
        with torch.no_grad():
            x = torch.as_tensor(obs).unsqueeze(0)
            action = agent.actor_mean(x).squeeze(0).numpy()
        action = np.clip(action, -1.0, 1.0)

        if abs(pitch) > math.radians(TIP_THRESHOLD_DEG) or abs(roll) > math.radians(TIP_THRESHOLD_DEG):
            data.ctrl[wheel_motor_id] = 0.0
            data.ctrl[reaction_wheel_motor_id] = 0.0
        else:
            data.ctrl[wheel_motor_id] = action[0]
            data.ctrl[reaction_wheel_motor_id] = action[1]

        mujoco.mj_step(model, data)
        viewer.sync()

        steps += 1
        if steps % PRINT_EVERY == 0:
            clear_output(wait=True)
            print(f"pitch: {math.degrees(pitch):6.2f} deg   roll: {math.degrees(roll):6.2f} deg")

        # Slow the simulation down so we can see the robot behave in real time
        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)

pitch:  -0.11 deg   roll:  -0.05 deg
